# 03 — Retain Deep Dive

This notebook covers `retain()` in depth:
- Basic and advanced retain calls
- Context, timestamps, and metadata
- Batch retains for bulk loading
- Dry-run extraction to preview facts
- Async retains for high throughput
- Per-user memory isolation patterns

In [ ]:
from hindsight_client import Hindsight
from datetime import datetime, timezone
import asyncio

HINDSIGHT_URL = "http://localhost:8888"
BANK_ID = "tutorial-retain"

client = Hindsight(base_url=HINDSIGHT_URL)

# Create a fresh bank for this tutorial
client.create_bank(
    bank_id=BANK_ID,
    name="Retain Tutorial",
    retain_extraction_mode="detailed",
    reflect_mission="I track user preferences, technical decisions, and project details."
)
print(f"Bank '{BANK_ID}' ready.")

## 1. Basic retain() — The Essentials

In [ ]:
# Simplest form — just content
client.retain(
    bank_id=BANK_ID,
    content="Alice works at Google as a software engineer."
)
print("✓ Simple retain done")

In [ ]:
# Verify — recall shows the extracted, canonicalized fact
results = client.recall(bank_id=BANK_ID, query="Alice")
for r in results.results:
    print(f"[{r.type}] {r.text}")

## 2. Full retain() with All Parameters

In [ ]:
# Retain with every available parameter
client.retain(
    bank_id=BANK_ID,
    content="Alice got promoted to Senior Engineer after leading the Search migration project.",
    context="career-update",                     # Categorize this memory
    timestamp="2026-06-15T10:00:00Z",            # When the event happened
    document_id="conversation_2026_06_15",       # Group with related retains
    metadata={                                    # Custom key-value data
        "source": "slack",
        "channel": "engineering",
        "user_id": "alice"
    }
)
print("✓ Full retain with all parameters done")
print("  context: 'career-update' — helps categorize during recall")
print("  timestamp: '2026-06-15' — enables temporal queries")
print("  document_id: groups related retains together")
print("  metadata: user_id, source, channel for filtering")

## 3. Context: Categorizing Memories

Use `context` to label what kind of information this is. Makes filtering easier later.

In [ ]:
# Categorize different types of information
contextual_facts = [
    ("P0 bug in auth service — token refresh failing", "incident"),
    ("Rolled back deploy v3.2.1 due to auth issue", "deployment"),
    ("Team decided to add integration tests for auth flows", "decision"),
    ("User reported latency spike after 10k concurrent users", "bug_report"),
    ("Added Redis caching layer for session tokens", "architecture"),
]

for content, context in contextual_facts:
    client.retain(
        bank_id=BANK_ID,
        content=content,
        context=context
    )
    print(f"✓ [{context}] {content[:50]}...")

## 4. Timestamps: Enabling Temporal Reasoning

Timestamps tell Hindsight WHEN something happened (not when it was stored).
This enables queries like "what happened in March 2026?"

In [ ]:
# Build a timeline of project events
timeline = [
    ("Project kickoff meeting held", datetime(2026, 1, 15, tzinfo=timezone.utc)),
    ("Backend MVP completed: FastAPI + PostgreSQL", datetime(2026, 3, 1, tzinfo=timezone.utc)),
    ("First staging deployment on AWS ECS", datetime(2026, 4, 15, tzinfo=timezone.utc)),
    ("Production launch — v1.0", datetime(2026, 5, 1, tzinfo=timezone.utc)),
    ("First paying customer onboarded", datetime(2026, 5, 10, tzinfo=timezone.utc)),
    ("Reached 1000 DAU milestone", datetime(2026, 6, 1, tzinfo=timezone.utc)),
]

for content, ts in timeline:
    client.retain(
        bank_id=BANK_ID,
        content=content,
        timestamp=ts.isoformat(),
        context="project-timeline",
        document_id="project-alpha-timeline"
    )
    print(f"✓ [{ts.strftime('%Y-%m-%d')}] {content[:50]}...")

In [ ]:
# Now temporal queries work
results = client.recall(
    bank_id=BANK_ID,
    query="What happened in May 2026?"
)
print("Events in May 2026:")
for r in results.results:
    print(f"  • {r.text}")

## 5. Metadata: Per-User Memory Isolation

Metadata is the key to storing multiple users' memories in a single bank
while keeping them isolated during recall.

In [ ]:
# Store memories for Alice and Bob in the same bank
users = {
    "alice": [
        "Prefers dark mode in all UIs",
        "Uses VSCode with vim keybindings",
        "Favorite language is Rust",
        "Works on the Search infrastructure team",
    ],
    "bob": [
        "Prefers light mode with large fonts",
        "Uses IntelliJ IDEA",
        "Favorite language is Python",
        "Works on the ML platform team",
    ]
}

for user_id, preferences in users.items():
    for pref in preferences:
        client.retain(
            bank_id=BANK_ID,
            content=pref,
            metadata={"user_id": user_id},
            context="user-preference"
        )
print("✓ Stored preferences for Alice and Bob")

In [ ]:
# Recall Alice's preferences ONLY — Bob's are invisible
alice_results = client.recall(
    bank_id=BANK_ID,
    query="user preferences",
    metadata_filter={"user_id": "alice"}
)

print("Alice's preferences:")
for r in alice_results.results:
    print(f"  • {r.text}")

print()

# Bob's preferences — completely different results
bob_results = client.recall(
    bank_id=BANK_ID,
    query="user preferences",
    metadata_filter={"user_id": "bob"}
)

print("Bob's preferences:")
for r in bob_results.results:
    print(f"  • {r.text}")

## 6. Batch Retain — Bulk Loading

`retain_batch()` is much faster for loading existing data. It processes all items
in a single LLM call.

In [ ]:
# Load conversation history in bulk
conversation = [
    {"content": "User: What's the best way to deploy FastAPI?", "context": "question"},
    {"content": "Asst: Docker + AWS ECS is our standard approach for FastAPI.", "context": "answer"},
    {"content": "User: Can we use Kubernetes instead? We have an existing EKS cluster.", "context": "question"},
    {"content": "Asst: Yes, EKS works well. Use the AWS Load Balancer Controller for ingress.", "context": "answer"},
    {"content": "User: OK, let's go with EKS. Can you set up the deployment config?", "context": "decision"},
    {"content": "Asst: Sure. I'll create the k8s manifests for the FastAPI service with HPA.", "context": "commitment"},
]

result = client.retain_batch(
    bank_id=BANK_ID,
    items=conversation,
    document_id="conv_deployment_discussion",
    retain_async=False
)
print(f"✓ Batch processed: {result.items_count} items in one call")

## 7. Dry-Run Extraction — Preview Before Committing

See what facts Hindsight would extract — without storing anything.

In [ ]:
# Preview fact extraction without storing
preview = client.dry_run_extract(
    bank_id=BANK_ID,
    content="""
    The user's name is Dr. Sarah Chen. She's the VP of Engineering at TechCorp.
    Her team of 45 engineers works on the cloud infrastructure platform.
    She prefers async communication via Slack and hates unnecessary meetings.
    She's been at TechCorp for 8 years, previously at AWS for 5 years.
    Her primary technical interests are distributed systems and observability.
    """
)
print("Preview — candidate facts (not stored):")
print(preview)
# Note: the actual preview structure depends on the API version
# It shows candidate facts, entities, and relationships

## 8. Async Retain — Fire and Forget

For high-throughput scenarios, use `retain_async=True` to return immediately.

In [ ]:
# Async retain — returns immediately, processes in background
async def async_retain_example():
    client = Hindsight(base_url=HINDSIGHT_URL)
    
    # Fire multiple retains and don't wait
    tasks = []
    for i in range(10):
        result = await client.aretain(
            bank_id=BANK_ID,
            content=f"Log entry #{i}: API request processed in {i*10}ms",
            context="metrics",
            retain_async=True  # ← Fire and forget
        )
        tasks.append(result)
    
    print(f"✓ Submitted {len(tasks)} async retains")
    print("  Each returns: {'success': True, 'operation_id': 'op_...'}")
    
    # Check pending operations
    await asyncio.sleep(2)
    stats = client.get_bank_stats(bank_id=BANK_ID)
    print(f"  Pending: {stats.get('pending_operations', '?')}")
    
    await client.aclose()

# Run the async example
await async_retain_example()

## 9. Practical Pattern: Conversation Logger

A reusable function that retains both sides of a conversation turn.

In [ ]:
class ConversationLogger:
    """Logs conversation turns to Hindsight with proper structure."""
    
    def __init__(self, client, bank_id, session_id):
        self.client = client
        self.bank_id = bank_id
        self.session_id = session_id
        self.turn = 0
    
    def log_turn(self, user_message, agent_response):
        """Log one turn of the conversation."""
        self.turn += 1
        
        # Log user message
        self.client.retain(
            bank_id=self.bank_id,
            content=f"User: {user_message}",
            context="user-message",
            document_id=self.session_id,
            metadata={
                "session_id": self.session_id,
                "turn": str(self.turn),
                "role": "user"
            }
        )
        
        # Log agent response
        self.client.retain(
            bank_id=self.bank_id,
            content=f"Agent: {agent_response}",
            context="agent-response",
            document_id=self.session_id,
            metadata={
                "session_id": self.session_id,
                "turn": str(self.turn),
                "role": "agent"
            }
        )
        
        return self.turn

# Usage
logger = ConversationLogger(client, BANK_ID, "session-2026-06-22-001")

logger.log_turn(
    "What's the status of the database migration?",
    "The migration is 80% complete. 3 of 4 tables migrated. The users table is in progress."
)

logger.log_turn(
    "Any issues so far?",
    "One issue: the orders table had a schema mismatch. We fixed it by adding the missing tax_rate column."
)

logger.log_turn(
    "When will it be done?",
    "Estimated completion: 2 hours. The users table has 50M rows so it's the slowest."
)

print(f"✓ Logged {logger.turn} conversation turns")

## 10. Cleanup

In [ ]:
# client.delete_bank(bank_id=BANK_ID)
print("Bank preserved for the next notebook.")

## Summary

You learned:
1. **Basic retain** — simple content storage
2. **Full parameters** — context, timestamp, document_id, metadata
3. **Context** — categorizing memories for filtering
4. **Timestamps** — enabling temporal queries
5. **Metadata** — per-user memory isolation in a shared bank
6. **Batch retain** — bulk loading existing data
7. **Dry-run** — previewing extraction before committing
8. **Async retain** — fire-and-forget for high throughput

**Next:** [04_recall.ipynb](04_recall.ipynb) — master TEMPR retrieval